# Foundation C — Environmental Mass Symbolic Analysis

For each candidate action:
1. Canonically normalize
2. Derive effective mass scaling on FRW
3. Derive coupling scaling
4. Lock test (R = m/g)
5. FRW background evaluation

In [ ]:
import sympy as sp
from sympy import sqrt, symbols, simplify, diff, Rational, pi, oo, limit

# Common symbols
M_Pl = symbols('M_Pl', positive=True)
H = symbols('H', positive=True)  # Hubble parameter
R_curv = symbols('R', positive=True)  # Ricci scalar (positive in matter/DE era)

def detect_lock(m_expr, g_expr, params, model_name):
    """Detect whether mass and coupling are locked."""
    R_ratio = simplify(m_expr / g_expr)
    print(f'=== {model_name} ===')
    print(f'  m     = {m_expr}')
    print(f'  g_eff = {g_expr}')
    print(f'  R = m/g = {R_ratio}')
    print()
    dependent = []
    for p in params:
        dR = simplify(diff(R_ratio, p))
        if dR != 0:
            dependent.append(str(p))
            print(f'  dR/d({p}) != 0  [DEPENDS]')
        else:
            print(f'  dR/d({p}) = 0  [independent]')
    print()
    if not dependent:
        print('  VERDICT: LOCKED')
    else:
        print(f'  VERDICT: UNLOCKED (via {dependent})')
    print()
    return R_ratio

## Candidate A: Conformally Coupled Scalar

Action: S = ∫ [-½(∂φ)² - (1/12)Rφ² + (α/M_Pl)φT^μ_μ]

- Z = 1 (already canonical)
- m² = R/6 (conformal mass)
- g = α/M_Pl

In [ ]:
alpha = symbols('alpha', positive=True)
xi = Rational(1, 6)  # conformal value

# Candidate A
m_A = sqrt(xi * R_curv)  # = sqrt(R/6)
g_A = alpha / M_Pl

R_A = detect_lock(m_A, g_A, [alpha, M_Pl, R_curv], 'Candidate A: Conformal Scalar')

In [ ]:
# Candidate A: FRW evaluation
print('Candidate A: FRW Background Evaluation')
print('='*50)
print()

# FRW Ricci scalar: R = 8πG ρ (1 - 3w) = 3H²(1-3w)Ω_m + 12H²Ω_Λ
# Simplified for different epochs:
epochs = {
    'Radiation (w=1/3)': 0,
    'Matter (w=0)': 3,      # R = 3H²
    'de Sitter (w=-1)': 12,  # R = 12H²
    'Today (approx)': 9,     # R ≈ 9H₀²
}

for epoch, R_coeff in epochs.items():
    m_val = sqrt(xi * R_coeff) if R_coeff > 0 else 0
    print(f'  {epoch}: R = {R_coeff}H², m/H = {float(m_val):.3f}')

print()
print('m_eff ~ H throughout matter/DE era.')
print('Field mass tracks Hubble scale. This is quintessence.')

## Candidate B: PGT 0⁺ Torsion + ξR

m²_eff = M_Pl²/(16π|t₂|) + ξR
g = 1/(M_Pl√|t₂|)

In [ ]:
t2 = symbols('t_2', positive=True)
xi_B = symbols('xi', positive=True)

# PGT bare mass (from the lock)
m0_sq_B = M_Pl**2 / (16 * pi * t2)

# Total effective mass
m_B = sqrt(m0_sq_B + xi_B * R_curv)
g_B = 1 / (M_Pl * sqrt(t2))

R_B = detect_lock(m_B, g_B, [t2, xi_B, R_curv, M_Pl], 'Candidate B: PGT 0+ + xi*R')

In [ ]:
# Candidate B: Dominance analysis
print('Candidate B: Bare mass vs environmental correction')
print('='*50)
print()

# Ratio of environmental to bare mass
ratio_B = simplify(xi_B * R_curv / m0_sq_B)
print(f'xi*R / m0² = {ratio_B}')
print()
print('Numerically for t_2 ~ 1, xi ~ 1, R ~ H₀²:')
print('  xi*R / m0² ~ 16π * t_2 * xi * H₀² / M_Pl²')
print('             ~ 16π * 1 * 1 * (10⁻³³)² / (2.4×10¹⁸)² eV')
print('             ~ 10⁻¹²²')
print()
print('The environmental correction is 10¹²² times smaller than the bare mass.')
print('VERDICT: Environmental term IRRELEVANT. PGT lock dominates.')

## Candidate C: Geometric Symmetron

V(φ) = -½μ²φ² + ¼λφ⁴ + ½ξRφ²

Broken phase (ξR < μ²): φ₀ = √((μ² - ξR)/λ), m² = 2(μ² - ξR)
Symmetric phase (ξR > μ²): φ₀ = 0, m² = ξR - μ²

In [ ]:
mu, lam = symbols('mu lambda', positive=True)

# Broken phase
phi0_broken = sqrt((mu**2 - xi_B * R_curv) / lam)
m_C_broken = sqrt(2 * (mu**2 - xi_B * R_curv))
g_C_broken = alpha * phi0_broken / M_Pl  # coupling proportional to VEV

print('=== Candidate C: Geometric Symmetron (broken phase) ===')
print(f'  φ₀ = {phi0_broken}')
print(f'  m = {m_C_broken}')
print(f'  g = {g_C_broken}')
print()

R_C = detect_lock(m_C_broken, g_C_broken, [mu, lam, xi_B, R_curv, alpha, M_Pl],
                  'Candidate C: Symmetron (broken phase)')

In [ ]:
# Candidate C: Naturalness check
print('Candidate C: Naturalness of μ')
print('='*50)
print()
print('For symmetron transition at current epoch:')
print('  R_crit = μ²/ξ ~ H₀²')
print('  With ξ ~ O(1): μ ~ H₀ ~ 10⁻³³ eV')
print()
print('Radiative corrections to μ²:')
print('  δμ² ~ (α²/16π²) Λ_UV²')
print('  For α ~ 1/M_Pl, Λ_UV ~ M_Pl:')
print('  δμ² ~ M_Pl²/(16π²) ~ 10³⁶ eV²')
print('  Requires: μ² ~ H₀² ~ 10⁻⁶⁶ eV²')
print('  Fine-tuning: δμ²/μ² ~ 10¹⁰²')
print()
print('VERDICT: FAILS_NATURALNESS. μ ~ H₀ is not protected.')

## Candidate D: Weyl Scalar (Stückelberg mode)

After gauge fixing: identical to Candidate A with specific ξ.

In [ ]:
# Candidate D: Weyl scalar
# After Stückelberg decomposition:
# m² = c * R where c is O(1), determined by Weyl action parameters
# g ~ 1/M_Pl (from Weyl gauge coupling)

c_W = symbols('c_W', positive=True)  # O(1) coefficient from Weyl action

m_D = sqrt(c_W * R_curv)
g_D = 1 / M_Pl

R_D = detect_lock(m_D, g_D, [c_W, R_curv, M_Pl], 'Candidate D: Weyl Scalar')

In [ ]:
# Candidate D: Comparison with Candidate A
print('Comparison: Candidates A and D')
print('='*50)
print()
print(f'A: m = sqrt(R/6),  g = alpha/M_Pl')
print(f'D: m = sqrt(c_W*R), g = 1/M_Pl')
print()
print('A has ξ = 1/6 (conformal), α free.')
print('D has c_W from Weyl action, coupling fixed at 1/M_Pl.')
print()
print('Phenomenologically equivalent: both give m ~ H, g ~ 1/M_Pl.')
print('D has stronger motivation (gauge symmetry) but same predictions.')

## Candidate E: Curvature-Dependent Kinetic Normalization

Z_eff = Z_0 + δZ(R)
m = μ/√Z_eff,  g = g_0/√Z_eff

Lock test: R = m/g = μ/g_0 = constant. LOCKED.

In [ ]:
Z_0 = symbols('Z_0', positive=True)
delta_Z = symbols('delta_Z', positive=True)  # curvature-dependent piece
mu_E = symbols('mu_E', positive=True)
g0_E = symbols('g_0', positive=True)

Z_eff = Z_0 + delta_Z
m_E = mu_E / sqrt(Z_eff)
g_E = g0_E / sqrt(Z_eff)

R_E = detect_lock(m_E, g_E, [Z_0, delta_Z, mu_E, g0_E], 'Candidate E: Kinetic Mixing')

In [ ]:
# Verify: R = m/g is independent of Z_eff
R_ratio_E = simplify(m_E / g_E)
print(f'R = m/g = {R_ratio_E}')
print(f'dR/d(Z_0) = {simplify(diff(R_ratio_E, Z_0))}')
print(f'dR/d(delta_Z) = {simplify(diff(R_ratio_E, delta_Z))}')
print()
print('R = mu_E/g_0, independent of kinetic normalization.')
print('CONFIRMED: Curvature-dependent Z does NOT break the lock.')

## Combined Lock Test Summary

In [ ]:
print('='*70)
print('FOUNDATION C: LOCK TEST SUMMARY')
print('='*70)
print()
print(f'{"Candidate":<40} {"Lock Status":<20} {"m_eff on FRW"}')
print('-'*70)
print(f'{"A: Conformal scalar":<40} {"UNLOCKED":<20} {"~ H (relevant)"}')
print(f'{"B: PGT 0+ + xi*R":<40} {"UNLOCKED *":<20} {"~ M_Pl (irrelevant)"}')
print(f'{"C: Geometric symmetron":<40} {"UNLOCKED":<20} {"~ H (tuned mu)"}')
print(f'{"D: Weyl scalar":<40} {"UNLOCKED":<20} {"~ H (relevant)"}')
print(f'{"E: Kinetic mixing":<40} {"LOCKED":<20} {"N/A"}')
print()
print('* Candidate B is formally unlocked by ξR but the correction is')
print('  10¹²⁰ times smaller than the bare PGT mass. Effectively locked.')
print()
print('SURVIVING: A, C, D (all produce m ~ H on FRW)')
print('ELIMINATED: B (bare mass dominates), E (lock preserved)')
print()
print('KEY OBSERVATION: All surviving candidates produce the SAME')
print('cosmological dynamics: a scalar with m ~ H. This is quintessence.')
print('The geometric origin does not modify the FRW phenomenology.')